# SZ native-channel sequence repair audit
20260616 full-day replay of 000555, 000937, 300179 from unchanged original Parquet. Full-market files were scanned; only three securities were reconstructed. This is not full-market acceptance. See run_sz_sequence_repair_validation.py for the executable command and receipt.


In [ ]:
import json
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'reports').is_dir() else Path.cwd().parent
out = root / 'reports/20260907-sz-sequence-repair'
read = lambda name: json.loads((out / name).read_text())
r = read('20260616-sz-three-symbols.json')
run = read('run.json')
assert run['exit_code'] == 0 and run['inputs_unchanged']
assert r['matched'] == r['comparable_anchors'] == r['total_anchors'] == 14078
assert all(r[k] == 0 for k in ['mismatched', 'not_comparable', 'excluded_by_status', 'data_errors', 'missing_source'])
assert not r['diagnostic_window_override']
assert r['continuous_lookahead_ms_by_symbol'] == {'000555': 1000, '000937': 1000, '300179': 3000}
replay = r['replay']
assert replay['applied_events'] == replay['selected_rows'] == 641540
assert replay['input_rows'] == replay['selected_rows'] + replay['excluded_rows']
assert replay['symbols'] == 3 and replay['channels'] == 1
assert replay['sz_sequence_regressions'] == [dict(channel=2015, stream='orders', previous_sequence=21524148, sequence=21524032, previous_source_row=66683411, source_row=66683412)]
assert replay['sz_sequence_repairs'] == [dict(channel=2015, stream='orders', rows=335214, natural_runs=2, merge_passes=1)]
print('Native inversion repaired; all 641540 selected events applied; 14078 snapshot checks matched.')


In [ ]:
t = read('20260616-sz-three-symbols-timings.json')['stages']
assert abs(t['restore_total_seconds'] + t['validation_total_seconds'] + t['unattributed_seconds'] - t['profiled_total_seconds']) < 1e-6
for key in ['restore_total_seconds', 'validation_total_seconds', 'profiled_total_seconds']: print(key, round(t[key], 3))
print('Driver elapsed includes polling tail; use benchmark clocks for timing.')
